# Merge Stage 1 -- Build Panel: Stock Daily

## Purpose
Combines three daily stock-level cleaned datasets into a single panel keyed on `(permno, date)`, filtered to only in-universe stock-days. This is the first merge step and produces the stock-level daily panel (Panel A).

## Sources (All Cleaned)
- `Data/Data_Collection/Cleaned/06_Daily_CRSP_Stock_Data/crsp_daily_clean.parquet` -- prices, returns, volume, market cap (the spine)
- `Data/Data_Collection/Cleaned/08_TAQ_Millisecond/taq_daily_clean.parquet` -- microstructure factors
- `Data/Data_Collection/Cleaned/10_OptionMetrics/om_options_factors_panel_clean.parquet` -- options factors
- `Data/Data_Collection/Cleaned/01_Top100_SP500_Universe/universe_annual_clean.parquet` -- defines which (permno, year) pairs are in-universe

## Merge Logic

### Step 1: Load Universe
The set of valid `(permno, year)` pairs is loaded from `universe_annual_clean.parquet`.

### Step 2: Load and Filter CRSP (Spine)
CRSP daily data is loaded and filtered to only rows where `(permno, year)` is in the universe. CRSP serves as the spine -- if a stock has no CRSP row on a day, it does not exist in the panel. A `bid_ask_spread` factor is computed as `abs(dlyask - dlybid) / midpoint`. Metadata columns (`ticker`, `primaryexch`, `year`) are dropped.

Column roles are defined:
- **Weight:** `dlycap` (market capitalisation, kept for cap-weighting in Stage 2 aggregation)
- **Target:** `dlyret` (daily return, separated from predictors -- used to compute the target, not as a feature)
- **Factors:** all remaining CRSP columns (prices, ex-dividend return, volume, dollar volume, OHLC, bid, ask, shares outstanding, plus computed bid-ask spread)

### Step 3: Load and Join TAQ
TAQ daily data is left-joined onto the CRSP spine on `(permno, date)`. Column name conflicts with CRSP are resolved by adding a `taq_` prefix. The join is verified to produce no row explosion (row count must equal the CRSP spine count). TAQ match rate is reported.

### Step 4: Load and Join OptionMetrics
OptionMetrics options panel is left-joined onto the panel on `(permno, date)`. Column name conflicts are resolved by adding an `om_` prefix. Row count is verified stable. OptionMetrics match rate is reported.

### Step 5: Final Column Inventory
All columns are categorised into ID (`permno`, `date`), weight (`dlycap`), target (`dlyret`), and factors. Factor count is broken down by source (CRSP, TAQ, OptionMetrics).

### Step 6: Validation
- No duplicate `(permno, date)` rows
- Row count, PERMNO count, date range, and unique date count
- Rows per year with approximate stocks-per-day
- Stocks per day distribution (min, mean, max)
- NaN summary by source (average NaN rate for CRSP, TAQ, OptionMetrics factor groups)
- Weight column (`dlycap`) checked for NaN, zeros, and range
- Target column (`dlyret`) checked for NaN, range, and mean

## Key Design Decisions
- **CRSP is the spine:** only stock-days with a CRSP row exist in the panel
- **TAQ and OptionMetrics are left-joined:** not all stocks have options or TAQ data on every day
- **`dlyret` is separated as target:** it is not a predictor feature
- **`dlycap` is kept:** needed for cap-weighting during Stage 2 cross-sectional aggregation
- **No winsorisation, no z-standardisation:** deferred to Stage 2
- **No forward-fill:** this is stock-level data where NaN is meaningful (stock had no options data that day, etc.)

## Output
`Data/Data_Collection/Final/Stage_1/panel_stock_daily.parquet` -- keyed on `(permno, date)`, containing ID columns, weight column, target column, and all stock-level daily factors from CRSP, TAQ, and OptionMetrics

In [1]:
# %% [markdown]
# # Merge Pipeline — Notebook 01: Build Panel A (Stock Daily)
#
# Combines three daily stock-level cleaned datasets into a single panel
# keyed on (permno, date), filtered to only in-universe stock-days.
#
# Sources:
#   - CRSP daily (spine) — prices, returns, volume, market cap
#   - TAQ daily — microstructure factors
#   - OptionMetrics — options factors
#
# Output: Data/Data_Collection/Final/step1_panels/panel_stock_daily.parquet
#
# Key decisions:
#   - CRSP is the spine: if a stock has no CRSP row on a day, it doesn't exist
#   - TAQ and OptionMetrics are left-joined (not all stocks have options/TAQ data)
#   - dlyret is separated: it's used to compute the target, not as a predictor
#   - dlycap is kept: needed for cap-weighting in Step 2 aggregation
#   - No winsorisation, no z-standardisation — those happen in Step 2
#   - No forward-fill — this is stock-level data

# %%
import pandas as pd
import numpy as np
from pathlib import Path
import gc

# ── Paths ────────────────────────────────────────────────────────────────────
CLEANED = Path('../../../Data/Data_Collection/Cleaned')
OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_1')
OUT_DIR.mkdir(parents=True, exist_ok=True)

UNIVERSE_ANNUAL = CLEANED / '01_Top100_SP500_Universe' / 'universe_annual_clean.parquet'
CRSP_PATH       = CLEANED / '06_Daily_CRSP_Stock_Data' / 'crsp_daily_clean.parquet'
TAQ_PATH        = CLEANED / '08_TAQ_Millisecond' / 'taq_daily_clean.parquet'
OM_PATH         = CLEANED / '10_OptionMetrics' / 'om_options_factors_panel_clean.parquet'

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: LOAD UNIVERSE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STEP 1: LOAD UNIVERSE")
print("=" * 90)

universe = pd.read_parquet(UNIVERSE_ANNUAL)
valid_pairs = set(zip(universe['permno'], universe['year']))
print(f"\n  Universe: {len(universe):,} (permno, year) pairs")
print(f"  Unique PERMNOs: {universe['permno'].nunique()}")
print(f"  Year range: {universe['year'].min()} – {universe['year'].max()}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: LOAD AND FILTER CRSP (SPINE)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 2: LOAD AND FILTER CRSP (SPINE)")
print("=" * 90)

crsp = pd.read_parquet(CRSP_PATH)
crsp['date'] = pd.to_datetime(crsp['date'])
print(f"\n  Raw CRSP: {len(crsp):,} rows × {crsp.shape[1]} columns")
print(f"  PERMNOs: {crsp['permno'].nunique()}")

# Filter to universe
crsp['_year'] = crsp['date'].dt.year
n_before = len(crsp)
crsp['_in_universe'] = crsp.apply(
    lambda r: (r['permno'], r['_year']) in valid_pairs, axis=1
)
crsp = crsp[crsp['_in_universe']].drop(columns=['_in_universe']).reset_index(drop=True)
print(f"  Filtered to universe: {n_before:,} → {len(crsp):,} rows")
print(f"  PERMNOs: {crsp['permno'].nunique()}")
print(f"  Date range: {crsp['date'].min().date()} → {crsp['date'].max().date()}")

# Identify CRSP column roles
crsp_meta = ['ticker', 'primaryexch', 'year', '_year']
crsp_weight = ['dlycap']           # kept for cap-weighting in aggregation
crsp_target = ['dlyret']           # separated — used for target, not as predictor
crsp_size = ['shrout']             # shares outstanding
crsp_factors = [c for c in crsp.columns
                if c not in ['permno', 'date'] + crsp_meta + crsp_weight + crsp_target]

print(f"\n  CRSP columns:")
print(f"    Weight: {crsp_weight}")
print(f"    Target: {crsp_target}")
print(f"    Factors: {len(crsp_factors)} — {crsp_factors}")

# Drop metadata columns we don't need going forward
crsp = crsp.drop(columns=[c for c in crsp_meta if c in crsp.columns], errors='ignore')

# Compute bid-ask spread from CRSP
if all(c in crsp.columns for c in ['dlyask', 'dlybid']):
    midpoint = (crsp['dlyask'] + crsp['dlybid']) / 2
    crsp['bid_ask_spread'] = (crsp['dlyask'] - crsp['dlybid']).abs() / midpoint.replace(0, np.nan)
    print(f"\n  Computed bid_ask_spread: "
          f"median={crsp['bid_ask_spread'].median():.6f}, "
          f"mean={crsp['bid_ask_spread'].mean():.6f}")

print(f"\n  CRSP spine: {len(crsp):,} rows × {crsp.shape[1]} columns")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: LOAD AND JOIN TAQ
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 3: LOAD AND JOIN TAQ")
print("=" * 90)

taq = pd.read_parquet(TAQ_PATH)
taq['date'] = pd.to_datetime(taq['date'])

# Drop TAQ metadata columns
taq_meta = [c for c in ['year'] if c in taq.columns]
taq = taq.drop(columns=taq_meta, errors='ignore')

taq_factor_cols = [c for c in taq.columns if c not in ['permno', 'date']]
print(f"\n  TAQ: {len(taq):,} rows × {len(taq_factor_cols)} factors")
print(f"  PERMNOs: {taq['permno'].nunique()}")

# Check for column name conflicts with CRSP
crsp_cols = set(crsp.columns) - {'permno', 'date'}
taq_cols = set(taq_factor_cols)
overlap = crsp_cols & taq_cols
if overlap:
    print(f"\n  ⚠ Column name overlap with CRSP: {overlap}")
    print(f"    Adding 'taq_' prefix to TAQ columns")
    taq = taq.rename(columns={c: f'taq_{c}' for c in overlap})
    taq_factor_cols = [c for c in taq.columns if c not in ['permno', 'date']]

# Left join TAQ onto CRSP spine
n_before = len(crsp)
panel = crsp.merge(taq, on=['permno', 'date'], how='left')
del taq
gc.collect()

# Verify no row explosion
assert len(panel) == n_before, (
    f"Row explosion after TAQ join: {n_before:,} → {len(panel):,}"
)

# Report match rate
n_taq_matched = panel[taq_factor_cols[0]].notna().sum()
print(f"\n  After TAQ join: {len(panel):,} rows")
print(f"  TAQ match rate: {n_taq_matched:,} / {len(panel):,} "
      f"({n_taq_matched/len(panel)*100:.1f}%)")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: LOAD AND JOIN OPTIONMETRICS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 4: LOAD AND JOIN OPTIONMETRICS")
print("=" * 90)

om = pd.read_parquet(OM_PATH)
om['date'] = pd.to_datetime(om['date'])

om_factor_cols = [c for c in om.columns if c not in ['permno', 'date']]
print(f"\n  OptionMetrics: {len(om):,} rows × {len(om_factor_cols)} factors")
print(f"  PERMNOs: {om['permno'].nunique()}")

# Check for column name conflicts
panel_cols = set(panel.columns) - {'permno', 'date'}
om_cols = set(om_factor_cols)
overlap = panel_cols & om_cols
if overlap:
    print(f"\n  ⚠ Column name overlap: {overlap}")
    print(f"    Adding 'om_' prefix to OptionMetrics columns")
    om = om.rename(columns={c: f'om_{c}' for c in overlap})
    om_factor_cols = [c for c in om.columns if c not in ['permno', 'date']]

# Left join OptionMetrics onto panel
n_before = len(panel)
panel = panel.merge(om, on=['permno', 'date'], how='left')
del om
gc.collect()

assert len(panel) == n_before, (
    f"Row explosion after OM join: {n_before:,} → {len(panel):,}"
)

n_om_matched = panel[om_factor_cols[0]].notna().sum()
print(f"\n  After OM join: {len(panel):,} rows")
print(f"  OM match rate: {n_om_matched:,} / {len(panel):,} "
      f"({n_om_matched/len(panel)*100:.1f}%)")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: FINAL COLUMN INVENTORY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 5: FINAL COLUMN INVENTORY")
print("=" * 90)

# Categorise all columns
id_cols = ['permno', 'date']
weight_cols = ['dlycap']
target_cols = ['dlyret']
factor_cols = [c for c in panel.columns if c not in id_cols + weight_cols + target_cols]

print(f"\n  ID columns: {id_cols}")
print(f"  Weight columns: {weight_cols}")
print(f"  Target columns: {target_cols}")
print(f"  Factor columns: {len(factor_cols)}")

# Breakdown by source
crsp_factor_list = [c for c in crsp_factors if c in panel.columns] + ['bid_ask_spread']
taq_factor_list = [c for c in taq_factor_cols if c in panel.columns]
om_factor_list = [c for c in om_factor_cols if c in panel.columns]
other = [c for c in factor_cols if c not in crsp_factor_list + taq_factor_list + om_factor_list]

print(f"\n  Factor breakdown by source:")
print(f"    CRSP:          {len(crsp_factor_list):>4d} factors")
print(f"    TAQ:           {len(taq_factor_list):>4d} factors")
print(f"    OptionMetrics: {len(om_factor_list):>4d} factors")
if other:
    print(f"    Other:         {len(other):>4d} — {other}")
print(f"    ────────────────────")
print(f"    Total:         {len(factor_cols):>4d} factors")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: VALIDATION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 6: VALIDATION")
print("=" * 90)

# ── 6a. No duplicate (permno, date) ─────────────────────────────────────────
n_dupes = panel.duplicated(subset=['permno', 'date']).sum()
print(f"\n  Duplicate (permno, date): {n_dupes}")
assert n_dupes == 0, f"Found {n_dupes} duplicates!"

# ── 6b. Row count sanity ────────────────────────────────────────────────────
print(f"\n  Total rows: {len(panel):,}")
print(f"  PERMNOs: {panel['permno'].nunique()}")
print(f"  Date range: {panel['date'].min().date()} → {panel['date'].max().date()}")
print(f"  Unique dates: {panel['date'].nunique():,}")

# ── 6c. Rows per year ───────────────────────────────────────────────────────
print(f"\n  Rows per year:")
rows_year = panel.groupby(panel['date'].dt.year).size()
for y, n in rows_year.items():
    avg = n / 252
    print(f"    {y}: {n:>7,d} rows (~{avg:.0f} stocks/day)")

# ── 6d. Stocks per day ──────────────────────────────────────────────────────
stocks_per_day = panel.groupby('date')['permno'].nunique()
print(f"\n  Stocks per day:")
print(f"    Mean: {stocks_per_day.mean():.1f}")
print(f"    Min:  {stocks_per_day.min()} (on {stocks_per_day.idxmin().date()})")
print(f"    Max:  {stocks_per_day.max()} (on {stocks_per_day.idxmax().date()})")

# ── 6e. NaN summary by source ───────────────────────────────────────────────
print(f"\n  NaN summary by source:")
for label, cols in [('CRSP', crsp_factor_list),
                     ('TAQ', taq_factor_list),
                     ('OptionMetrics', om_factor_list)]:
    present_cols = [c for c in cols if c in panel.columns]
    if not present_cols:
        continue
    nan_rate = panel[present_cols].isna().mean().mean() * 100
    n_nan = panel[present_cols].isna().sum().sum()
    total = len(panel) * len(present_cols)
    print(f"    {label:<15s} {nan_rate:>5.2f}% avg NaN  "
          f"({n_nan:,} / {total:,} cells)")

# ── 6f. Weight column check ─────────────────────────────────────────────────
print(f"\n  Weight column (dlycap):")
print(f"    NaN: {panel['dlycap'].isna().sum()}")
print(f"    Zero: {(panel['dlycap'] == 0).sum()}")
print(f"    Range: [{panel['dlycap'].min():,.0f}, {panel['dlycap'].max():,.0f}]")

# ── 6g. Target column check ─────────────────────────────────────────────────
print(f"\n  Target column (dlyret):")
print(f"    NaN: {panel['dlyret'].isna().sum()}")
print(f"    Range: [{panel['dlyret'].min():.6f}, {panel['dlyret'].max():.6f}]")
print(f"    Mean: {panel['dlyret'].mean():.6f}")

# ── 6h. Sample ──────────────────────────────────────────────────────────────
print(f"\n  Sample (first 3 rows, selected columns):")
sample_cols = ['permno', 'date', 'dlyret', 'dlycap', 'dlyprc',
               'dlyretx', 'dlyvol', 'bid_ask_spread']
sample_cols = [c for c in sample_cols if c in panel.columns]
print(panel[sample_cols].head(3).to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 7: SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 7: SAVE")
print("=" * 90)

# Sort for clean output
panel = panel.sort_values(['permno', 'date']).reset_index(drop=True)

out_path = OUT_DIR / 'panel_stock_daily.parquet'
panel.to_parquet(out_path, index=False, engine='pyarrow')

print(f"\n  ✓ Saved: {out_path}")
print(f"    {len(panel):,} rows × {panel.shape[1]} columns")
print(f"    ID: permno, date")
print(f"    Weight: dlycap")
print(f"    Target: dlyret")
print(f"    Factors: {len(factor_cols)}")
print(f"    Size: {out_path.stat().st_size / 1e9:.2f} GB")

print("\nPanel A (stock daily) complete.")

STEP 1: LOAD UNIVERSE

  Universe: 2,100 (permno, year) pairs
  Unique PERMNOs: 227
  Year range: 2004 – 2024

STEP 2: LOAD AND FILTER CRSP (SPINE)

  Raw CRSP: 1,010,280 rows × 19 columns
  PERMNOs: 227
  Filtered to universe: 1,010,280 → 525,957 rows
  PERMNOs: 227
  Date range: 2004-01-02 → 2024-12-31

  CRSP columns:
    Weight: ['dlycap']
    Target: ['dlyret']
    Factors: 12 — ['dlyprc', 'dlyretx', 'dlyreti', 'dlyvol', 'dlyopen', 'dlyhigh', 'dlylow', 'dlyclose', 'dlybid', 'dlyask', 'dlyprcvol', 'shrout']

  Computed bid_ask_spread: median=0.000219, mean=0.000379

  CRSP spine: 525,957 rows × 17 columns

STEP 3: LOAD AND JOIN TAQ

  TAQ: 978,371 rows × 192 factors
  PERMNOs: 227

  After TAQ join: 525,957 rows
  TAQ match rate: 514,247 / 525,957 (97.8%)

STEP 4: LOAD AND JOIN OPTIONMETRICS

  OptionMetrics: 930,755 rows × 31 factors
  PERMNOs: 216

  After OM join: 525,957 rows
  OM match rate: 499,378 / 525,957 (94.9%)

STEP 5: FINAL COLUMN INVENTORY

  ID columns: ['permno', 'd